<a href="https://colab.research.google.com/github/NilsChabloz/2026_Intro_Python/blob/main/live/part-I/1.7-oop-for-environmental-systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.7-oop-for-environmental-systems.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [1]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# 1.7) Object-Oriented and Defensive Programming

Some data naturally travels with the operations that act on it: a lake has a name, a surface area, a maximum depth, and a growing list of temperature readings, and the things you do with it — add a reading, compute its mean, estimate the energy needed to warm it — belong to that lake. A class bundles such state and behaviour into one object. This subchapter builds a `Lake` from Switzerland's lakes, composes lakes into a register, contrasts a lightweight `dataclass` record, and is careful about the opposite lesson too: when a plain function is the better tool. It then turns from writing code to trusting it: `assert` for internal invariants versus exceptions for bad input, validating physical preconditions, logging instead of printing, and testing with pytest — the concrete habits for checking code you did not write yourself, including code an AI assistant generated for you.

**🎯 Learning objectives**

- Recognise the classes behind pandas, numpy and geopandas objects, and describe a class as one level of abstraction.
- Define a class with `__init__` and `self`, create instances, and validate arguments in the constructor.
- Distinguish instance attributes (unique per object) from class attributes (shared).
- Write type-hinted methods that read state and methods that mutate it.
- Recognise dunder methods, and write a useful `__repr__` and `__eq__`.
- Use a dataclass for a lightweight record, and composition to build larger objects from smaller ones.
- Extend a class with inheritance, calling `super().__init__` to reuse the parent's constructor.
- Judge when a class earns its keep (state that persists, shared by several operations) and when a function is clearer.
- Use assert to check invariants that should hold if the code is correct.
- Signal and handle bad input with exceptions: raise, custom exception types, and try/except/else/finally.
- Validate physical preconditions explicitly.
- Prefer logging to print for messages that carry a severity level.
- Write and run a pytest suite with plain tests, a fixture, and parametrisation.

## 1.7.1 Object-Oriented Programming

### Classes You Already Use

When you write `df.groupby("lake")["temp_celsius"].mean()`, you work at a high level of abstraction: you say what you want, and the details of how it happens stay out of sight. Underneath, a `DataFrame` object stores its columns in numpy arrays, a numpy array is a block of numbers in memory with a description of how to index it, and that block is processed by compiled code. Each level offers a small set of named operations and hides the machinery below it, so you can use the level you are on without knowing the one beneath. Choosing the level is the skill: too low and every analysis re-implements array loops; too high and you cannot say anything the library did not anticipate.

A **class** is how Python builds one such level. Every object used so far in the book is an instance of a class somebody else wrote: `np.array` creates an instance of `numpy.ndarray`, `pd.read_csv` returns a `DataFrame`, and a `GeoDataFrame` from geopandas is a `DataFrame` that has been extended with a geometry column (the inheritance pattern met later in this subchapter). Even `LinearRegression().fit(...)` in 1.8 is a method call on an instance. What changes in this subchapter is that you write the class yourself.

In [2]:
import numpy as np
import pandas as pd

temps_celsius = np.array([6.1, 12.4, 21.0])
table = pd.DataFrame({"temp_celsius": temps_celsius})

print(type(temps_celsius))
print(type(table))
# the same method name on two different classes, each with its own implementation
print(temps_celsius.mean(), table["temp_celsius"].mean())

<class 'numpy.ndarray'>
<class 'pandas.core.frame.DataFrame'>
13.166666666666666 13.166666666666666


### Classes and Instances

A **class** is the blueprint that defines a new kind of object: what data it carries (its *attributes*) and what it can do (its *methods*). An **instance** is one specific object built from that blueprint — `geneva` and `zurich` below are two different instances of the same `Lake` class, each with its own name, area, depth, and list of readings.

`__init__` is the *constructor*: the method Python calls automatically each time a new instance is created, and its job is to set up that instance's starting state. Inside any method, `self` refers to the particular instance the method was called on — it is how `geneva.add_reading(6.1)` updates only `geneva`'s readings, and `zurich.add_reading(5.2)` only `zurich`'s.

#### A class to represent a lake

The smallest useful class stores a single attribute:

In [3]:
class Lake:
    def __init__(self, name: str):
        self.name = name

An instance is created by calling the class like a function, with the arguments `__init__` expects after `self`:

In [4]:
zurich = Lake("Zurich")
print(zurich, zurich.name)

<__main__.Lake object at 0x7d72c35456a0> Zurich


Printing the object itself shows only its class and a memory address; `zurich.name` is the data it holds. A lake needs more than a name. The version below adds a surface area and a maximum depth (rounded values, in km² and m) and a list for readings, and checks the numbers as the object is built. Every lake passes through `__init__`, so a check there means no lake with an impossible depth can exist.

In [5]:
class Lake:
    def __init__(self, name: str, area_km2: float, max_depth_m: float):
        if area_km2 <= 0:
            raise ValueError(f"area must be positive, got {area_km2} km²")
        # Lake Baikal, the deepest lake on Earth, reaches 1642 m: anything outside is a data-entry error
        if max_depth_m <= 0 or max_depth_m > 1642:
            raise ValueError(f"implausible maximum depth {max_depth_m} m")
        self.name = name                                  # instance attributes: per-object
        self.area_km2 = area_km2
        self.max_depth_m = max_depth_m
        self.surface_temps_celsius: list[float] = []      # each lake owns its own list


geneva = Lake("Geneva", 580, 310)
print(geneva.name, geneva.area_km2, geneva.max_depth_m, geneva.surface_temps_celsius)

Geneva 580 310 []


In [6]:
# 3100 m is a typo for 310 m; the constructor refuses it
Lake("Geneva", 580, 3100)

ValueError: implausible maximum depth 3100 m

#### Magic (dunder) methods

Methods whose names begin and end with a double underscore — *dunder* methods, also called magic or special methods — are not usually called by name. Python calls them itself, in response to built-in operations, and defining one customises how instances of the class behave. `__init__` is the first: Python calls it whenever a new instance is created. Two more belong right next to it:

- `__repr__` is called whenever Python needs a printable representation of an object, as `print` does. Without it you get the class name and memory address seen above. Inside an f-string, `!r` inserts the `repr` of a value instead of its plain text, so a string keeps its quotes: `'Geneva'` rather than `Geneva`.
- `__eq__` is called by `==`. Without it, two instances are equal only if they are the very same object, so two separately built `Lake("Geneva", 580, 310)` would compare as different. `isinstance(other, Lake)` checks that the right-hand side is a `Lake`; returning `NotImplemented` otherwise lets Python try the comparison the other way round instead of raising an error.

The full list is in the [Python data model](https://docs.python.org/3/reference/datamodel.html#special-method-names).

In [8]:
class Lake:
    def __init__(self, name: str, area_km2: float, max_depth_m: float):
        if area_km2 <= 0:
            raise ValueError(f"area must be positive, got {area_km2} km²")
        # Lake Baikal, the deepest lake on Earth, reaches 1642 m: anything outside is a data-entry error
        if max_depth_m <= 0 or max_depth_m > 1642:
            raise ValueError(f"implausible maximum depth {max_depth_m} m")
        self.name = name                                  # instance attributes: per-object
        self.area_km2 = area_km2
        self.max_depth_m = max_depth_m
        self.surface_temps_celsius: list[float] = []      # each lake owns its own list

    def __repr__(self) -> str:
        return (f"Lake({self.name!r}, {self.area_km2} km², max depth {self.max_depth_m} m, "
                f"n={len(self.surface_temps_celsius)})")

    def __eq__(self, other) -> bool:
        # two Lake objects describe the same lake if name, area and depth agree;
        # the readings collected so far are deliberately left out of the comparison
        if not isinstance(other, Lake):
            return NotImplemented
        return (self.name, self.area_km2, self.max_depth_m) == (other.name, other.area_km2, other.max_depth_m)

In [9]:
geneva = Lake("Geneva", 580, 310)
zurich = Lake("Zurich", 88, 136)

print(geneva)                                  # __repr__ in action
print(zurich)
print(geneva == Lake("Geneva", 580, 310))      # __eq__: same lake, built twice
print(geneva == zurich)                        # different lakes

Lake('Geneva', 580 km², max depth 310 m, n=0)
Lake('Zurich', 88 km², max depth 136 m, n=0)
True
False


Next, the methods: functions defined inside the class, which act on one instance through `self`. Some read the instance's state, some change it, and one below also uses *class* attributes, values defined once on the class and shared by every lake.

In [10]:
# the built-in sum() adds up the elements of an iterable
print(sum([1.0, 2.0, 3.0]))

6.0


In [11]:
class Lake:
    # class attributes: one value shared by every lake (approximate constants for fresh water)
    density_kg_m3 = 1000.0
    heat_capacity_j_kg_k = 4186.0

    def __init__(self, name: str, area_km2: float, max_depth_m: float):
        if area_km2 <= 0:
            raise ValueError(f"area must be positive, got {area_km2} km²")
        # Lake Baikal, the deepest lake on Earth, reaches 1642 m: anything outside is a data-entry error
        if max_depth_m <= 0 or max_depth_m > 1642:
            raise ValueError(f"implausible maximum depth {max_depth_m} m")
        self.name = name                                  # instance attributes: per-object
        self.area_km2 = area_km2
        self.max_depth_m = max_depth_m
        self.surface_temps_celsius: list[float] = []      # each lake owns its own list

    def __repr__(self) -> str:
        return (f"Lake({self.name!r}, {self.area_km2} km², max depth {self.max_depth_m} m, "
                f"n={len(self.surface_temps_celsius)})")

    def __eq__(self, other) -> bool:
        # two Lake objects describe the same lake if name, area and depth agree;
        # the readings collected so far are deliberately left out of the comparison
        if not isinstance(other, Lake):
            return NotImplemented
        return (self.name, self.area_km2, self.max_depth_m) == (other.name, other.area_km2, other.max_depth_m)

    def add_reading(self, temp_celsius: float) -> None:
        # a method that MUTATES state
        self.surface_temps_celsius.append(temp_celsius)

    def mean_temperature(self) -> float:
        # a method that READS state
        if not self.surface_temps_celsius:
            raise ValueError(f"{self.name} has no readings")
        return sum(self.surface_temps_celsius) / len(self.surface_temps_celsius)

    def energy_to_warm_joules(self, delta_celsius: float, layer_depth_m: float) -> float:
        # uses the shared class attributes together with instance state
        volume_m3 = self.area_km2 * 1e6 * layer_depth_m
        return volume_m3 * self.density_kg_m3 * self.heat_capacity_j_kg_k * delta_celsius

In [12]:
geneva = Lake("Geneva", 580, 310)
zurich = Lake("Zurich", 88, 136)

for t in [6.1, 12.4, 21.0]:
    geneva.add_reading(t)
for t in [5.2, 13.0, 22.8]:
    zurich.add_reading(t)

print(geneva.surface_temps_celsius)
print("Geneva mean:", round(geneva.mean_temperature(), 2), "°C")
# energy to warm the top 10 m of the lake by 1 °C, in petajoules (1 PJ = 1e15 J)
print("Geneva, 10 m layer, +1 °C:", round(geneva.energy_to_warm_joules(1.0, 10.0) / 1e15, 1), "PJ")

[6.1, 12.4, 21.0]
Geneva mean: 13.17 °C
Geneva, 10 m layer, +1 °C: 24.3 PJ


### Instance versus Class Attributes

An *instance* attribute belongs to one object (each lake's own `surface_temps_celsius`). A *class* attribute is shared by every instance (the single `density_kg_m3` and `heat_capacity_j_kg_k`). Shared **constants** are a good use of class attributes; shared **mutable** state can lead to bugs.

In [13]:
# the constants are single values, reachable via the class or any instance
print(Lake.density_kg_m3,
      geneva.density_kg_m3,
      zurich.density_kg_m3)

# the readings are independent per instance
print("Geneva readings:", geneva.surface_temps_celsius)
print("Zurich readings:", zurich.surface_temps_celsius)

1000.0 1000.0 1000.0
Geneva readings: [6.1, 12.4, 21.0]
Zurich readings: [5.2, 13.0, 22.8]


### dataclasses: Records with Less Ceremony

When an object is mostly a bundle of fields, `@dataclass` generates `__init__`, `__repr__`, and `__eq__` for you from type-annotated attributes — the same three methods written by hand for `Lake` above.

In [14]:
from dataclasses import dataclass

@dataclass
class Reading:
    timestamp: str
    temp_celsius: float

r = Reading("2024-06-01", 18.2)
print(r)                                                  # auto __repr__
print(r.temp_celsius)
print(Reading("2024-06-01", 18.2) == Reading("2024-06-01", 18.2))   # auto __eq__

Reading(timestamp='2024-06-01', temp_celsius=18.2)
18.2
True


<details>
<summary><b>🔍 Going deeper: composition, building larger objects from smaller ones</b></summary>

Composition is a *has-a* relationship: a `LakeRegister` *has* lakes. The container delegates work to the objects it holds rather than re-implementing it.

```python
class LakeRegister:
    def __init__(self, name: str):
        self.name = name
        self.lakes: dict[str, Lake] = {}

    def add_lake(self, lake: Lake) -> None:
        self.lakes[lake.name] = lake

    def warmest_lake(self) -> Lake:
        # delegate to each lake's own mean_temperature method
        warmest = None
        for lake in self.lakes.values():   # .values() gives the dict's values: the lake objects
            if warmest is None or lake.mean_temperature() > warmest.mean_temperature():
                warmest = lake
        return warmest

    def __repr__(self) -> str:
        return f"LakeRegister({self.name!r}, {len(self.lakes)} lakes)"

register = LakeRegister("Switzerland")
register.add_lake(geneva)
register.add_lake(zurich)
print(register)
print("warmest:", register.warmest_lake().name)
```

</details>

**🧠 Computational-thinking fundamental: reach for a class only when there is state to bundle**

A class earns its keep when data and the operations on it belong together and the object carries state between calls — a station accumulating readings, a model remembering what it learned. When there is no state to keep, an object is just ceremony around a function. Converting celsius to kelvin needs no class: there is nothing to remember between calls, so a plain function does the job with less code. Wrapping a stateless calculation in a class anyway adds a constructor to write, an object to instantiate, and attributes to track for data that never changes — more code to read and maintain for no behaviour gained.

### When to Use a Class

The case for a class is the mirror image of the case against: state that outlives a single call, several operations that read or change that state, and a need for the state to stay valid as a whole. When all three hold, a class keeps the pieces together and checks them once, in `__init__`. A running mean over a buoy's temperature stream shows all three. The buoy reports one value at a time and the full record is never stored, so the count and the running total must be remembered between calls. As functions, every call would have to receive both numbers and return updated ones, and nothing would stop the caller from passing the count of one buoy with the total of another.

In [15]:
class RunningMean:
    def __init__(self):
        self.count = 0          # state that persists between calls
        self.total_celsius = 0.0

    def update(self, temp_celsius: float) -> None:
        self.count += 1
        self.total_celsius += temp_celsius

    def mean(self) -> float:
        if self.count == 0:
            raise ValueError("no values yet")
        return self.total_celsius / self.count

In [16]:
buoy = RunningMean()
for t in [6.1, 12.4, 21.0]:      # values arrive one at a time
    buoy.update(t)
print("mean so far:", round(buoy.mean(), 2), "°C from", buoy.count, "values")

mean so far: 13.17 °C from 3 values


### When Not to Use a Class

A stateless transformation needs no object. Wrapping a one-line conversion in a class adds boilerplate and hides a simple function behind a constructor.

In [17]:
# a pure function is the right tool here: input in, output out, no state
def celsius_to_kelvin(temp_celsius: float) -> float:
    return temp_celsius + 273.15

print(celsius_to_kelvin(18.2))
# a class with no attributes and a single method would only add ceremony

291.34999999999997


### Inheritance: Extending a Class

A subclass inherits a parent's attributes and methods — an *is-a* relationship, as opposed to composition's *has-a*. Calling `super().__init__` reuses the parent's constructor, so the subclass only has to set up what is new.

In [18]:
class Reservoir(Lake):
    def __init__(self, name: str, area_km2: float, max_depth_m: float):
        super().__init__(name, area_km2, max_depth_m)   # reuse the parent's constructor
        self.outflow_m3s: list[float] = []              # extra state specific to a reservoir

    def add_outflow(self, q_m3s: float) -> None:
        self.outflow_m3s.append(q_m3s)

gruyere = Reservoir("Lac de la Gruyère", 9.6, 75)
gruyere.add_reading(9.8)          # inherited from Lake
gruyere.add_outflow(45.0)
print(gruyere.name, gruyere.surface_temps_celsius, gruyere.outflow_m3s)
print("mean temperature:", round(gruyere.mean_temperature(), 2))   # inherited method

Lac de la Gruyère [9.8] [45.0]
mean temperature: 9.8


Prefer composition to deep inheritance hierarchies; inherit only for a genuine *is-a* relationship.

<details>
<summary><b>🔍 Going deeper: property validation</b></summary>

A `@property` lets an attribute run validation on assignment while still being accessed like plain data.

```python
class Lake:
    def __init__(self, max_depth_m):
        self.max_depth_m = max_depth_m       # goes through the setter

    @property
    def max_depth_m(self):
        return self._max_depth_m

    @max_depth_m.setter
    def max_depth_m(self, value):
        if value <= 0:
            raise ValueError("maximum depth must be positive")
        self._max_depth_m = value
```

</details>

<details>
<summary><b>🔍 Going deeper: abstract base classes</b></summary>

An abstract base class defines an interface that subclasses must implement, enforced at instantiation.

```python
from abc import ABC, abstractmethod

class Sensor(ABC):
    @abstractmethod
    def read(self) -> float:
        ...
# a subclass that does not implement read() cannot be instantiated
```

ABCs document the contract a family of objects must satisfy.

</details>

<details>
<summary><b>🔍 Going deeper: the scikit-learn estimator-as-object pattern</b></summary>

scikit-learn models are objects that learn state in `fit` (stored on attributes with a trailing underscore) and use it in `predict` — the bridge to the machine-learning subchapter.

```python
class MeanRegressor:
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)     # learned state
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]
```

Every estimator you will meet follows this fit/predict object shape.

</details>

## 1.7.2 Defensive Programming

### From Designing Objects to Trusting Code

The first half of this subchapter was about building your own objects. The second is a different skill: reading and stress-testing code you did not write yourself — including code generated by an AI assistant, which can run and look plausible while still being wrong on exactly the input you didn't try. `assert`, exceptions, explicit validation, logging, and tests are the concrete tools for that: ways to make a piece of code state its own assumptions and get caught the moment those assumptions break, rather than trusting that code runs correctly just because it runs.

### assert for Invariants

An `assert` documents and checks a condition that should always be true *if the code is correct*. It is a statement about the program's own logic, not about external input — and it is removed when Python runs with the `-O` flag.

In [19]:
def normalise(weights: list[float]) -> list[float]:
    total = sum(weights)
    result = []
    for w in weights:
        result.append(round(w / total, 2))
    # invariant: normalised weights sum to 1 (a check on our own arithmetic)
    assert abs(sum(result) - 1.0) < 1e-9, f"normalised weights sum to {sum(result)}, not 1"
    return result

print(normalise([1.0, 3.0]))

[0.25, 0.75]


This call passes. Three equal weights expose a mistake in the function itself: rounding each share to two decimals gives 0.33 three times, and the shares no longer sum to 1.

In [20]:
normalise([1.0, 1.0, 1.0])

AssertionError: normalised weights sum to 0.99, not 1

The input was valid; the bug is the `round` inside the function, which belongs at display time, not in the calculation. The assert caught it at the point where the invariant broke, instead of letting shares that sum to 0.99 flow into later results.

### Exceptions: raise, Custom Types, and try/except/else/finally

An exception signals a runtime problem. `raise` triggers one; a custom exception subclass names a specific failure; `try/except/else/finally` handles it — `else` runs when no exception occurred, `finally` always runs.

In [21]:
class PhysicalRangeError(ValueError):
    # a named failure mode, more specific than a bare ValueError
    pass

In [22]:
def to_kelvin(temp_celsius: float) -> float:
    if temp_celsius < -273.15:
        raise PhysicalRangeError(f"{temp_celsius} °C is below absolute zero")
    return temp_celsius + 273.15

In [23]:
for value in [25.0, -300.0]:
    try:
        kelvin = to_kelvin(value)
    except PhysicalRangeError as err:
        print(f"rejected, the value input is {value}: {err}")
    else:
        # runs only when no exception was raised
        print(f"ok, the value input is {value}, the kelvin is {round(kelvin, 2)}")
    finally:
        print("checked", value)               # always runs

ok, the value input is 25.0, the kelvin is 298.15
checked 25.0
rejected, the value input is -300.0: -300.0 °C is below absolute zero
checked -300.0


### Validating Physical Preconditions

External input — a file, a user value, a network response — must be validated with exceptions, not asserts, because it can be wrong even when the code is correct.

In [24]:
def validate_observation(temp_celsius: float, discharge_m3s: float) -> None:
    if temp_celsius < -273.15:
        raise PhysicalRangeError("temperature below absolute zero")
    if discharge_m3s < 0.0:
        raise ValueError("discharge cannot be negative")

validate_observation(18.0, 45.0)          # valid input passes silently
try:
    validate_observation(18.0, -5.0)
except ValueError as err:
    print("caught:", err)

caught: discharge cannot be negative


### Logging over print

`print` writes unconditionally to stdout. `logging` attaches a severity level to each message, so the same code can be verbose while debugging and quiet in production, and can route messages to files or services without edits.

In [25]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
logger = logging.getLogger("mlees")

logger.debug("suppressed: below the INFO threshold")
logger.info("loaded 45 readings")
logger.warning("3 temperature values missing")

INFO: loaded 45 readings


### Testing with pytest

pytest discovers functions named `test_*`, runs them, and reports failures with readable assertion output. A *fixture* supplies reusable setup; *parametrisation* runs one test over many cases. Here we write a small module and its test file to disk, then run the suite with `subprocess.run`, which starts pytest as a separate process (`sys.executable -m pytest` uses the same Python as the notebook) and, with `capture_output=True, text=True`, collects what it prints as strings.

In [26]:
from pathlib import Path

In [27]:
Path("_files").mkdir(exist_ok=True)

In [28]:
module_src = """
def to_kelvin(temp_celsius):
    if temp_celsius < -273.15:
        raise ValueError("below absolute zero")
    return temp_celsius + 273.15
"""
Path("_files/thermo.py").write_text(module_src, encoding="utf-8")

142

In [29]:
test_src = """
import pytest
from thermo import to_kelvin

def test_freezing_point():
    assert to_kelvin(0.0) == 273.15

@pytest.fixture
def boiling_celsius():
    return 100.0

def test_with_fixture(boiling_celsius):
    assert to_kelvin(boiling_celsius) == 373.15

@pytest.mark.parametrize("celsius, kelvin", [(0.0, 273.15), (-273.15, 0.0), (25.0, 298.15)])
def test_conversions(celsius, kelvin):
    assert to_kelvin(celsius) == pytest.approx(kelvin)

def test_below_absolute_zero_raises():
    with pytest.raises(ValueError):
        to_kelvin(-300.0)
"""
Path("_files/test_thermo.py").write_text(test_src, encoding="utf-8")
print("wrote thermo.py and test_thermo.py")

wrote thermo.py and test_thermo.py


In [30]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_thermo.py", "-q", "--color=no"],
    capture_output=True, text=True, cwd="_files",
)
# stderr is printed too, so that a failure to start pytest is visible instead of silent
print((result.stdout + result.stderr).strip())

......                                                                   [100%]
6 passed in 0.04s


## *When generated code lies: the shared class-level list*

Asked for a lake class, an assistant declares the readings list at the class level. That single list is then shared by every instance — the object-oriented version of the mutable-default-argument bug.

In [31]:
class LakeBuggy:
    surface_temps_celsius = []        # class attribute: ONE list shared by all instances

    def __init__(self, name):
        self.name = name

    def add_reading(self, temp_celsius):
        self.surface_temps_celsius.append(temp_celsius)

a = LakeBuggy("A")
b = LakeBuggy("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.surface_temps_celsius)   # expected [10.0]
print("B readings:", b.surface_temps_celsius)   # expected [20.0]

A readings: [10.0, 20.0]
B readings: [10.0, 20.0]


**⚠️ Diagnosis: mutable state declared on the class is shared**

`surface_temps_celsius = []` sits on the class, so there is exactly one list and every instance appends to it; lake A and lake B end up with each other's data, silently. Mutable per-instance state must be created inside `__init__` with `self.surface_temps_celsius = []`, so each object gets its own list. Class attributes are for values that are genuinely shared and, ideally, immutable.

In [32]:
class LakeFixed:
    def __init__(self, name):
        self.name = name
        self.surface_temps_celsius = []      # per-instance list, created on construction

    def add_reading(self, temp_celsius):
        self.surface_temps_celsius.append(temp_celsius)

a = LakeFixed("A")
b = LakeFixed("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.surface_temps_celsius, "| B readings:", b.surface_temps_celsius)

A readings: [10.0] | B readings: [20.0]


**📌 Takeaways**

- Every pandas, numpy or geopandas object is an instance of a class: a class is one level of abstraction, with its machinery hidden behind named methods.
- A class bundles state (attributes) and behaviour (methods); `__init__` stores data on `self` at construction.
- Validate arguments in `__init__`, so that an object with impossible values can never be built.
- Instance attributes are per-object; class attributes are shared — good for constants, dangerous for mutable state.
- Dunder methods such as `__init__`, `__repr__` and `__eq__` are called by Python itself; write a `__repr__` for readable objects and an `__eq__` when two instances can describe the same thing.
- Type-hint methods, and separate methods that read state from those that mutate it.
- Use a `dataclass` for plain records, and composition to assemble larger objects from smaller ones.
- A subclass inherits a parent's attributes and methods; call `super().__init__` to reuse its constructor, then add what's new.
- A class earns its keep when state persists between calls, several operations share it, and it must stay valid as a whole.
- A stateless transformation should be a function, not a class.
- Never put mutable state (`readings = []`) on the class body: it is shared across all instances. Initialise it in `__init__`.
- `assert` checks internal invariants and is stripped by `python -O`; never use it to validate external input.
- Raise exceptions (a custom subclass when it clarifies intent) for bad input; handle with try/except/else/finally.
- Validate physical preconditions explicitly, and fail early and loudly.
- Prefer `logging` to `print`: messages gain a severity level and can be filtered or redirected.
- Test with pytest — plain tests, fixtures for setup, parametrisation for many cases.

## Summary

| Concept | Rule to remember |
|---|---|
| Classes | Bundle state (attributes) with behaviour (methods); `__init__` stores data on `self`. |
| Attributes | Instance attributes are per-object, class attributes are shared — never put mutable state on the class body. |
| Records | Use a `dataclass` for plain records, and composition to assemble larger objects from smaller ones. |
| Inheritance | A subclass extends a parent; `super().__init__` reuses its constructor. |
| When to | State that persists between calls, shared by several operations, and valid as a whole: a class. |
| When not to | A stateless transformation should be a function, not a class. |
| `assert` | For internal invariants only — `python -O` strips it, so never validate external input with it. |
| Exceptions | `raise` for bad input, `try`/`except`/`else`/`finally` to handle it; fail early and loudly. |
| Logging | Prefer `logging` to `print`: messages gain a severity level and can be filtered or redirected. |
| Testing | pytest — plain tests, fixtures for setup, parametrisation for many cases. |

## Resources

- [Object-Oriented Programming (OOP) in Python](https://realpython.com/python3-object-oriented-programming/) — classes, instances, attributes, methods, and inheritance, with worked examples.
- [Python Classes: The Power of Object-Oriented Programming](https://realpython.com/python-classes/) — instance vs class attributes, dataclasses, abstract base classes, and explicit guidance on when *not* to use a class.
- [pytest documentation](https://docs.pytest.org/en/stable/) — writing tests, fixtures, parametrisation, and assertions.